# Variant 11 — Answer-only LoRA + Model-only Voting

Notebook này là bản sửa từ Variant 11 theo hướng an toàn hơn với luật:

- Fine-tune `NlpHUST/gpt2-vietnamese` theo dạng **answer-only**.
- Không dùng rule solver / Python solver để tự giải toán từ `query_vi`.
- Inference chỉ dùng output do model sinh ra: greedy trước, sampling nhẹ nếu greedy không ra đáp án hợp lệ.
- Python chỉ dùng cho: đọc dữ liệu, clean dữ liệu, extract đáp án từ output, tính validation score, ghi file JSON.
- Khi ghi `model_output`, đáp án cuối lấy từ candidate do model generate, không lấy từ công thức/rule tự code.

Mục tiêu của variant này: giữ pipeline gần Variant 11 nhưng loại bỏ phần rủi ro về luật, vẫn tạo được `valid_output.json`, `valid_report.json` và `test_predictions.json`.


## Cell 1 — Setup, config, paths

In [1]:

import os
import re
import gc
import json
import math
import random
import inspect
import warnings
from pathlib import Path
from collections import Counter, defaultdict
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

VARIANT_NAME = "v11_answer_only_lora_model_only"
SAFE_EOS_ID = 50256

# Kaggle official paths. Local fallback paths are included only for debugging outside Kaggle.
TRAIN_PATH_CANDIDATES = [
    Path("/kaggle/input/datasets/kimanh2002/dataset-math/train.json"),
    Path("/kaggle/input/dataset-math/train.json"),
    Path("/mnt/data/train.json"),
]
VALID_PATH_CANDIDATES = [
    Path("/kaggle/input/datasets/kimanh2002/dataset-math/valid.json"),
    Path("/kaggle/input/dataset-math/valid.json"),
    Path("/mnt/data/valid.json"),
]
TEST_PATH_CANDIDATES = [
    Path("/kaggle/input/datasets/kimanh2002/dataset-math/test.json"),
    Path("/kaggle/input/dataset-math/test.json"),
    Path("/mnt/data/test.json"),
]
MODEL_PATH_CANDIDATES = [
    Path("/kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese"),
    Path("/kaggle/input/nlphustgpt2-vietnamese"),
    Path("/kaggle/input/nlp-hust-gpt2-vietnamese"),
    Path("/kaggle/input/gpt2-vietnamese"),
]

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/mnt/data")
PROC_DIR = WORK_DIR / f"processed_{VARIANT_NAME}"
MODEL_OUT_DIR = WORK_DIR / f"lora_{VARIANT_NAME}"
PRED_DIR = WORK_DIR / f"predictions_{VARIANT_NAME}"
for p in [PROC_DIR, MODEL_OUT_DIR, PRED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Main knobs for <= 3 hours. Reduce TRAIN_MAX_SAMPLES / NUM_TRAIN_EPOCHS if your Kaggle GPU is slow.
TRAIN_MAX_SAMPLES = 40000
MAX_LENGTH = 256
GEN_MAX_NEW_TOKENS = 24
USE_SAMPLING_FALLBACK = True
NUM_SAMPLE_CANDIDATES = 3
GEN_TEMPERATURE = 0.7
GEN_TOP_P = 0.9

NUM_TRAIN_EPOCHS = 5
PER_DEVICE_TRAIN_BATCH_SIZE = 8
PER_DEVICE_EVAL_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03

LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.03

LOGGING_STEPS = 50
EVAL_STEPS = 700
SAVE_STEPS = 700
EVAL_LOSS_MAX_SAMPLES = 512
VALID_EVAL_MAX_SAMPLES = None  # None = full valid; set 300/500 for quick smoke test.

DO_TRAIN = True
DO_VALIDATE = True
DO_TEST_PREDICT = True

print("Variant:", VARIANT_NAME)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Variant: v11_answer_only_lora_model_only
CUDA: True
GPU: Tesla T4


## Cell 2 — I/O helpers and path discovery

In [2]:

def first_existing(paths: List[Path], required: bool = True, name: str = "path") -> Optional[Path]:
    for p in paths:
        if p.exists():
            print(f"{name}: {p}")
            return p
    if required:
        raise FileNotFoundError(f"Cannot find {name}. Tried: {[str(p) for p in paths]}")
    print(f"{name}: not found")
    return None

TRAIN_PATH = first_existing(TRAIN_PATH_CANDIDATES, True, "TRAIN_PATH")
VALID_PATH = first_existing(VALID_PATH_CANDIDATES, True, "VALID_PATH")
TEST_PATH = first_existing(TEST_PATH_CANDIDATES, False, "TEST_PATH")


def find_model_path() -> Path:
    for p in MODEL_PATH_CANDIDATES:
        if p.exists():
            print("MODEL_PATH:", p)
            return p
    # Fallback: search common Kaggle input subdirs for config.json.
    root = Path("/kaggle/input")
    if root.exists():
        for cfg in root.rglob("config.json"):
            parent = cfg.parent
            files = {x.name for x in parent.iterdir() if x.is_file()}
            if any(name.startswith("pytorch_model") or name.endswith(".safetensors") for name in files):
                print("MODEL_PATH auto-found:", parent)
                return parent
    raise FileNotFoundError("Cannot find local NlpHUST/gpt2-vietnamese model directory.")


def read_json_or_jsonl(path: Path) -> Any:
    text = path.read_text(encoding="utf-8-sig").strip()
    if not text:
        return []
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # JSONL fallback.
    records = []
    ok_jsonl = True
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            ok_jsonl = False
            break
    if ok_jsonl and records:
        return records

    # Concatenated JSON objects fallback.
    decoder = json.JSONDecoder()
    idx = 0
    records = []
    while idx < len(text):
        while idx < len(text) and text[idx].isspace():
            idx += 1
        if idx >= len(text):
            break
        obj, end = decoder.raw_decode(text, idx)
        records.append(obj)
        idx = end
    return records


def ensure_list_records(obj: Any) -> List[Dict[str, Any]]:
    if isinstance(obj, list):
        return [x for x in obj if isinstance(x, dict)]
    if isinstance(obj, dict):
        for key in ["data", "records", "items", "examples"]:
            if isinstance(obj.get(key), list):
                return [x for x in obj[key] if isinstance(x, dict)]
        return [obj]
    return []


def write_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def write_jsonl(records: List[Dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

raw_train = ensure_list_records(read_json_or_jsonl(TRAIN_PATH))
raw_valid = ensure_list_records(read_json_or_jsonl(VALID_PATH))
print("raw_train:", len(raw_train), "raw_valid:", len(raw_valid))
print("train sample keys:", raw_train[0].keys() if raw_train else None)


TRAIN_PATH: /kaggle/input/datasets/kimanh2002/dataset-math/train.json
VALID_PATH: /kaggle/input/datasets/kimanh2002/dataset-math/valid.json
TEST_PATH: not found
raw_train: 95400 raw_valid: 1000
train sample keys: dict_keys(['original_question_vi', 'original_question_en', 'query_vi', 'query_en', 'response_vi', 'response_en', 'type'])


## Cell 3 — Answer extraction and scoring

Extractor này ưu tiên đáp án sau anchor (`Đáp án là`, `Câu trả lời là`, `The answer is`, `####`) và xử lý cơ bản `\frac{a}{b}`, `a/b`, `a\pi`, decimal comma, thousand separator.

In [3]:

ANSWER_ANCHOR_PATTERNS = [
    r"Đáp án là\s*[:：]?\s*([^\n]+)",
    r"Câu trả lời là\s*[:：]?\s*([^\n]+)",
    r"The answer is\s*[:：]?\s*([^\n]+)",
    r"####\s*([^\n]+)",
]

BAD_ANSWER_TOKENS = ["nan", "inf", "undefined", "không xác định", "không đủ", "unknown", "none", "n/a"]


def normalize_unicode_text(text: str) -> str:
    text = str(text or "")
    text = text.replace("\ufeff", "").replace("\u200b", "").replace("\u00a0", " ")
    text = text.replace("−", "-").replace("–", "-").replace("—", "-")
    text = text.replace("×", "*").replace("÷", "/")
    text = text.replace("\\đóng hộp", "\\boxed").replace("\\ đóng hộp", "\\boxed")
    text = text.replace("\\dfrac", "\\frac").replace("\\tfrac", "\\frac")
    return text


def strip_asy_blocks(text: str) -> str:
    return re.sub(r"\[asy\].*?\[/asy\]", " ", str(text or ""), flags=re.I | re.S)


def unwrap_boxed_once(s: str) -> str:
    m = re.search(r"\\boxed\s*\{", s)
    if not m:
        return s
    start = m.end()
    depth = 1
    i = start
    while i < len(s):
        if s[i] == "{":
            depth += 1
        elif s[i] == "}":
            depth -= 1
            if depth == 0:
                return s[:m.start()] + s[start:i] + s[i+1:]
        i += 1
    return s


def strip_boxed(s: str) -> str:
    s = str(s or "")
    for _ in range(5):
        s2 = unwrap_boxed_once(s)
        if s2 == s:
            break
        s = s2
    return s


def clean_answer_candidate(ans: str) -> str:
    ans = normalize_unicode_text(ans)
    ans = strip_boxed(ans).strip()
    ans = ans.replace("$", "").replace("`", "").strip()
    ans = re.split(r"\s+(?:Đáp án là|Câu trả lời là|The answer is)\s*[:：]?", ans, flags=re.I)[0]
    ans = ans.split("\n")[0].strip()
    ans = re.sub(r"^[=:：\s]+", "", ans).strip()
    ans = re.sub(r"(?:\.|,|;|:|。)+$", "", ans).strip()
    ans = re.sub(r"\s+", " ", ans)

    # Prefer latex fraction if present.
    m = re.search(r"-?\\frac\s*\{\s*-?\d+(?:[\.,]\d+)?\s*\}\s*\{\s*-?\d+(?:[\.,]\d+)?\s*\}", ans)
    if m:
        return m.group(0).replace(" ", "")

    # Prefer tuple if present.
    m = re.search(r"\((-?\d+(?:[\.,]\d+)?\s*,\s*)+-?\d+(?:[\.,]\d+)?\)", ans)
    if m:
        return re.sub(r"\s+", "", m.group(0))

    # Prefer coefficient*pi form.
    m = re.search(r"-?\d+(?:[\.,]\d+)?\s*\\?pi", ans, flags=re.I)
    if m:
        return m.group(0).replace(" ", "")

    # Prefer common fraction.
    m = re.search(r"-?\d+(?:[\.,]\d+)?\s*/\s*-?\d+(?:[\.,]\d+)?", ans)
    if m:
        return m.group(0).replace(" ", "")

    # Otherwise last numeric token, with optional thousand separators.
    nums = re.findall(r"-?\d+(?:[\.,]\d+)*", ans)
    if nums:
        return nums[-1]
    return ans.strip()


def extract_final_answer(text: str, fallback_last_number: bool = True) -> Optional[str]:
    text = normalize_unicode_text(text)
    if not text.strip():
        return None
    matches = []
    for pat in ANSWER_ANCHOR_PATTERNS:
        for m in re.finditer(pat, text, flags=re.I):
            matches.append((m.start(), m.group(1)))
    if matches:
        matches.sort(key=lambda x: x[0])
        cand = clean_answer_candidate(matches[-1][1])
        if cand and not any(tok in cand.lower() for tok in BAD_ANSWER_TOKENS):
            return cand
    if fallback_last_number:
        cand = clean_answer_candidate(text)
        if cand and not any(tok in cand.lower() for tok in BAD_ANSWER_TOKENS):
            return cand
    return None


def _normalize_num_token(x: str) -> str:
    x = str(x).strip().replace(" ", "")
    # 40.320 or 1,234,567 => thousand separators.
    if re.fullmatch(r"-?\d{1,3}([\.,]\d{3})+", x):
        return x.replace(".", "").replace(",", "")
    # 4,5 => 4.5 decimal comma.
    if re.fullmatch(r"-?\d+,\d{1,6}", x):
        return x.replace(",", ".")
    return x.replace(",", "")


def parse_numeric_answer(ans: Any) -> Optional[float]:
    if ans is None:
        return None
    s = clean_answer_candidate(str(ans))
    s = normalize_unicode_text(s)
    s = s.replace("$", "").strip()

    # Latex fraction.
    m = re.fullmatch(r"(-?)\\frac\{\s*(-?\d+(?:[\.,]\d+)?)\s*\}\{\s*(-?\d+(?:[\.,]\d+)?)\s*\}", s)
    if m:
        sign = -1.0 if m.group(1) == "-" else 1.0
        a = float(_normalize_num_token(m.group(2)))
        b = float(_normalize_num_token(m.group(3)))
        return sign * a / b if b != 0 else None

    # Common fraction.
    m = re.fullmatch(r"(-?\d+(?:[\.,]\d+)?)\s*/\s*(-?\d+(?:[\.,]\d+)?)", s)
    if m:
        a = float(_normalize_num_token(m.group(1)))
        b = float(_normalize_num_token(m.group(2)))
        return a / b if b != 0 else None

    # pi forms: pi, -pi/2, 36\pi.
    pi_s = s.replace("\\pi", "pi").replace("π", "pi")
    m = re.fullmatch(r"(-?)(?:(\d+(?:[\.,]\d+)?))?\s*pi(?:\s*/\s*(\d+(?:[\.,]\d+)?))?", pi_s, flags=re.I)
    if m:
        sign = -1.0 if m.group(1) == "-" else 1.0
        coef = float(_normalize_num_token(m.group(2))) if m.group(2) else 1.0
        den = float(_normalize_num_token(m.group(3))) if m.group(3) else 1.0
        return sign * coef * math.pi / den if den != 0 else None

    # Plain number.
    if re.fullmatch(r"-?\d+(?:[\.,]\d+)*", s):
        try:
            return float(_normalize_num_token(s))
        except Exception:
            return None
    return None


def relative_error(pred: Any, gold: Any) -> Optional[float]:
    p = parse_numeric_answer(pred)
    g = parse_numeric_answer(gold)
    if p is None or g is None:
        return None
    return abs(p - g) / max(1.0, abs(g))


def score_one(pred: Any, gold: Any) -> int:
    # Exact string fallback for non-scalar answers.
    if pred is not None and gold is not None:
        if clean_answer_candidate(str(pred)) == clean_answer_candidate(str(gold)):
            return 10
    err = relative_error(pred, gold)
    if err is None:
        return 0
    if err <= 0.01:
        return 10
    if err <= 0.10:
        return 5
    if err <= 0.50:
        return 1
    return 0

# Quick sanity checks.
for s in ["Đáp án là: 37", "Câu trả lời là: \\frac{9}{20}", "The answer is: 36\\pi", "#### 40.320"]:
    a = extract_final_answer(s)
    print(s, "=>", a, "=>", parse_numeric_answer(a))


Đáp án là: 37 => 37 => 37.0
Câu trả lời là: \frac{9}{20} => \frac{9}{20} => 0.45
The answer is: 36\pi => 36\pi => 113.09733552923255
#### 40.320 => 40.320 => 40320.0


## Cell 4 — Preprocess to answer-only records

In [4]:

def get_query(raw: Dict[str, Any]) -> str:
    return normalize_unicode_text(str(raw.get("query_vi", "") or "")).strip()


def get_response(raw: Dict[str, Any]) -> str:
    return normalize_unicode_text(str(raw.get("response_vi", "") or "")).strip()


def get_type(raw: Dict[str, Any]) -> str:
    return str(raw.get("type", "UNKNOWN") or "UNKNOWN")


def make_prompt(query: str) -> str:
    return f"Câu hỏi: {query}\nĐáp án:"


def make_target(answer: str) -> str:
    return f" {answer}"


def normalize_query(q: str) -> str:
    q = strip_asy_blocks(q)
    q = normalize_unicode_text(q)
    q = re.sub(r"\s+", " ", q).strip()
    return q


def preprocess_records(raw_records: List[Dict[str, Any]], split: str, train_mode: bool = True) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    records = []
    drops = []
    seen = set()

    for i, raw in enumerate(raw_records):
        q = normalize_query(get_query(raw))
        resp = get_response(raw)
        typ = get_type(raw)
        ans = extract_final_answer(resp, fallback_last_number=False)

        reason = None
        if not q:
            reason = "missing_query"
        elif train_mode and not resp:
            reason = "missing_response"
        elif ans is None or not str(ans).strip():
            reason = "missing_answer"
        elif len(q) > 1400:
            reason = "query_too_long"
        elif any(tok in str(ans).lower() for tok in BAD_ANSWER_TOKENS):
            reason = "bad_answer_token"

        if reason:
            drops.append({"split": split, "raw_index": i, "reason": reason, "type": typ, "query_vi": q[:300]})
            continue

        key = (q, str(ans))
        if train_mode and key in seen:
            drops.append({"split": split, "raw_index": i, "reason": "duplicate_query_answer", "type": typ, "query_vi": q[:300]})
            continue
        seen.add(key)

        rec = {
            "id": raw.get("id", i),
            "raw_index": i,
            "type": typ,
            "query_vi": q,
            "final_answer": clean_answer_candidate(ans),
            "prompt": make_prompt(q),
            "target": make_target(clean_answer_candidate(ans)),
        }
        records.append(rec)

    return records, drops

train_records, train_drops = preprocess_records(raw_train, "train", train_mode=True)
valid_records, valid_drops = preprocess_records(raw_valid, "valid", train_mode=False)

# Shuffle and cap train set.
rng = random.Random(SEED)
rng.shuffle(train_records)
if TRAIN_MAX_SAMPLES is not None and len(train_records) > TRAIN_MAX_SAMPLES:
    train_records = train_records[:TRAIN_MAX_SAMPLES]

write_jsonl(train_records, PROC_DIR / "train_answer_only.jsonl")
write_jsonl(valid_records, PROC_DIR / "valid_answer_only.jsonl")
write_jsonl(train_drops + valid_drops, PROC_DIR / "drop_log.jsonl")

prep_report = {
    "variant": VARIANT_NAME,
    "target_format": "answer_only",
    "counts": {
        "raw_train": len(raw_train),
        "raw_valid": len(raw_valid),
        "train_records": len(train_records),
        "valid_records": len(valid_records),
        "train_dropped": len(train_drops),
        "valid_dropped": len(valid_drops),
    },
    "train_type_distribution": dict(Counter(r["type"] for r in train_records)),
    "valid_type_distribution": dict(Counter(r["type"] for r in valid_records)),
    "drop_summary": dict(Counter(d["reason"] for d in train_drops + valid_drops)),
}
write_json(prep_report, PROC_DIR / "preprocess_report.json")
print(json.dumps(prep_report, ensure_ascii=False, indent=2)[:2000])

pd.DataFrame(train_records[:5])[['type','query_vi','final_answer','prompt','target']]


{
  "variant": "v11_answer_only_lora_model_only",
  "target_format": "answer_only",
  "counts": {
    "raw_train": 95400,
    "raw_valid": 1000,
    "train_records": 40000,
    "valid_records": 997,
    "train_dropped": 37558,
    "valid_dropped": 3
  },
  "train_type_distribution": {
    "GSM_AnsAug": 4401,
    "MATH_FOBAR": 1683,
    "GSM_FOBAR": 5331,
    "GSM_SV": 6344,
    "GSM_Rephrased": 11815,
    "MATH_Rephrased": 5888,
    "MATH_AnsAug": 2964,
    "MATH_SV": 1574
  },
  "valid_type_distribution": {
    "GSM_Rephrased": 197,
    "MATH_Rephrased": 115,
    "MATH_SV": 41,
    "GSM_AnsAug": 209,
    "GSM_SV": 97,
    "GSM_FOBAR": 122,
    "MATH_AnsAug": 171,
    "MATH_FOBAR": 45
  },
  "drop_summary": {
    "duplicate_query_answer": 37460,
    "missing_answer": 95,
    "query_too_long": 6
  }
}


,type,query_vi,final_answer,prompt,target
0,GSM_AnsAug,Một thẩm phán giám sát mười bảy vụ án. Hai ngư...,4,Câu hỏi: Một thẩm phán giám sát mười bảy vụ án...,4
1,MATH_FOBAR,"Giả sử X là thừa số của $a$, $a$ là ước của $1...",3,"Câu hỏi: Giả sử X là thừa số của $a$, $a$ là ư...",3
2,GSM_FOBAR,"Randy đạt 90, 98, x và 94 trong bốn câu hỏi đầ...",92,"Câu hỏi: Randy đạt 90, 98, x và 94 trong bốn c...",92
3,GSM_AnsAug,Kim gọi một bữa ăn giá 10 USD và một đồ uống g...,5,Câu hỏi: Kim gọi một bữa ăn giá 10 USD và một ...,5
4,GSM_SV,"John có x miếng kẹo cao su, Cole có 45 miếng k...",54,"Câu hỏi: John có x miếng kẹo cao su, Cole có 4...",54


## Cell 5 — Load tokenizer/model and build torch datasets

In [5]:

from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_PATH = find_model_path()

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(str(MODEL_PATH), local_files_only=True)

# Force safe eos/pad ids required by the competition note.
tokenizer.pad_token_id = SAFE_EOS_ID
tokenizer.eos_token_id = SAFE_EOS_ID
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token if tokenizer.eos_token is not None else "<|endoftext|>"
model.config.pad_token_id = SAFE_EOS_ID
model.config.eos_token_id = SAFE_EOS_ID

print("vocab_size:", len(tokenizer), "pad:", tokenizer.pad_token_id, "eos:", tokenizer.eos_token_id)

class AnswerOnlyDataset(Dataset):
    def __init__(self, records: List[Dict[str, Any]], tokenizer, max_length: int):
        self.records = records
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        r = self.records[idx]
        prompt = r["prompt"]
        target = r["target"] + (self.tokenizer.eos_token or "")

        prompt_ids = self.tokenizer(prompt, add_special_tokens=False)["input_ids"]
        full = self.tokenizer(
            prompt + target,
            add_special_tokens=False,
            truncation=True,
            max_length=self.max_length,
        )
        input_ids = full["input_ids"]
        attention_mask = full["attention_mask"]
        labels = input_ids.copy()
        prompt_len = min(len(prompt_ids), len(labels))
        labels[:prompt_len] = [-100] * prompt_len

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

class CausalLMCollator:
    def __init__(self, tokenizer, label_pad_token_id: int = -100):
        self.tokenizer = tokenizer
        self.label_pad_token_id = label_pad_token_id

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)
        batch = {"input_ids": [], "attention_mask": [], "labels": []}
        for f in features:
            pad_len = max_len - len(f["input_ids"])
            batch["input_ids"].append(torch.cat([f["input_ids"], torch.full((pad_len,), self.tokenizer.pad_token_id, dtype=torch.long)]))
            batch["attention_mask"].append(torch.cat([f["attention_mask"], torch.zeros(pad_len, dtype=torch.long)]))
            batch["labels"].append(torch.cat([f["labels"], torch.full((pad_len,), self.label_pad_token_id, dtype=torch.long)]))
        return {k: torch.stack(v) for k, v in batch.items()}

train_dataset = AnswerOnlyDataset(train_records, tokenizer, MAX_LENGTH)
eval_loss_records = valid_records[:min(EVAL_LOSS_MAX_SAMPLES, len(valid_records))]
eval_dataset = AnswerOnlyDataset(eval_loss_records, tokenizer, MAX_LENGTH)
collator = CausalLMCollator(tokenizer)

print("train_dataset:", len(train_dataset), "eval_dataset:", len(eval_dataset))


MODEL_PATH: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
h.{0...11}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab_size: 50258 pad: 50256 eos: 50256
train_dataset: 40000 eval_dataset: 512


## Cell 6 — Apply LoRA

In [6]:

try:
    from peft import LoraConfig, get_peft_model, TaskType
except Exception as e:
    raise ImportError(
        "PEFT is required for this LoRA variant. On Kaggle, attach a dataset/package with peft if needed, still with Internet OFF."
    ) from e

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["c_attn", "c_proj", "c_fc"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

if torch.cuda.is_available():
    model = model.cuda()


trainable params: 4,718,592 || all params: 129,158,400 || trainable%: 3.6533


## Cell 7 — Fine-tune answer-only LoRA

In [7]:

from transformers import Trainer, TrainingArguments


def build_training_args() -> TrainingArguments:
    raw_kwargs = dict(
        output_dir=str(MODEL_OUT_DIR),
        overwrite_output_dir=True,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        learning_rate=LEARNING_RATE,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        logging_steps=LOGGING_STEPS,
        save_steps=SAVE_STEPS,
        eval_steps=EVAL_STEPS,
        save_total_limit=2,
        fp16=torch.cuda.is_available(),
        report_to="none",
        dataloader_num_workers=2,
        remove_unused_columns=False,
        load_best_model_at_end=False,
    )
    sig = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" in sig:
        raw_kwargs["eval_strategy"] = "steps"
    elif "evaluation_strategy" in sig:
        raw_kwargs["evaluation_strategy"] = "steps"

    kwargs = {k: v for k, v in raw_kwargs.items() if k in sig}
    dropped = sorted(set(raw_kwargs) - set(kwargs))
    if dropped:
        print("Dropped unsupported TrainingArguments:", dropped)
    return TrainingArguments(**kwargs)


def build_trainer_kwargs(args: TrainingArguments) -> Dict[str, Any]:
    base = dict(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=collator,
    )
    sig = inspect.signature(Trainer.__init__).parameters
    return {k: v for k, v in base.items() if k in sig}

if DO_TRAIN:
    training_args = build_training_args()
    trainer = Trainer(**build_trainer_kwargs(training_args))
    train_result = trainer.train()
    print(train_result)
    trainer.save_model(str(MODEL_OUT_DIR))
    tokenizer.save_pretrained(str(MODEL_OUT_DIR))
    write_json({"train_result": str(train_result), "variant": VARIANT_NAME}, MODEL_OUT_DIR / "train_result.json")
else:
    print("DO_TRAIN=False, skip training")

model.eval()
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Dropped unsupported TrainingArguments: ['overwrite_output_dir']


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
700,2.065566,2.082193
1400,1.893428,1.988187
2100,1.743253,1.954440
2800,1.686842,1.934367


TrainOutput(global_step=3125, training_loss=2.0443805444335936, metrics={'train_runtime': 4644.4031, 'train_samples_per_second': 43.063, 'train_steps_per_second': 0.673, 'total_flos': 1.4811885515390976e+16, 'train_loss': 2.0443805444335936, 'epoch': 5.0})


## Cell 8 — Inference policy: model-only

Cell này thay thế hoàn toàn rule solver của Variant 11. Notebook không định nghĩa hàm giải toán từ đề bằng regex/công thức. Các bước hợp lệ ở inference chỉ là generate bằng model, extract đáp án từ output, vote giữa các candidate do model sinh ra, và format lại JSON submission.


In [8]:
INFERENCE_POLICY = {
    "variant": VARIANT_NAME,
    "model_only": True,
    "no_rule_solver": True,
    "no_python_math_solver_from_query": True,
    "allowed_python_usage": [
        "data_loading",
        "data_cleaning",
        "answer_extraction_from_model_output",
        "local_validation_scoring",
        "json_export",
    ],
    "candidate_sources": ["model_greedy", "model_sample"] if USE_SAMPLING_FALLBACK else ["model_greedy"],
}
print(json.dumps(INFERENCE_POLICY, ensure_ascii=False, indent=2))


{
  "variant": "v11_answer_only_lora_model_only",
  "model_only": true,
  "no_rule_solver": true,
  "no_python_math_solver_from_query": true,
  "allowed_python_usage": [
    "data_loading",
    "data_cleaning",
    "answer_extraction_from_model_output",
    "local_validation_scoring",
    "json_export"
  ],
  "candidate_sources": [
    "model_greedy",
    "model_sample"
  ]
}


## Cell 9 — Model generation, model-only voting and final prediction functions


In [9]:

@torch.no_grad()
def generate_answer_only(prompt: str, do_sample: bool = False, num_return_sequences: int = 1) -> List[str]:
    model.eval()
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(model.device)

    gen_kwargs = dict(
        max_new_tokens=GEN_MAX_NEW_TOKENS,
        pad_token_id=SAFE_EOS_ID,
        eos_token_id=SAFE_EOS_ID,
        num_return_sequences=num_return_sequences,
    )
    if do_sample:
        gen_kwargs.update(dict(do_sample=True, temperature=GEN_TEMPERATURE, top_p=GEN_TOP_P))
    else:
        gen_kwargs.update(dict(do_sample=False, num_beams=1))

    out = model.generate(**inputs, **gen_kwargs)
    decoded = tokenizer.batch_decode(out, skip_special_tokens=True)

    answers = []
    for text in decoded:
        # Remove prompt prefix if model repeats it. This does not solve the problem;
        # it only extracts the answer generated by the model.
        tail = text[len(prompt):] if text.startswith(prompt) else text.split("Đáp án:")[-1]
        ans = extract_final_answer(tail, fallback_last_number=True)
        if ans is None:
            ans = clean_answer_candidate(tail)
        ans = clean_answer_candidate(ans)
        if ans:
            answers.append(ans)
    return answers


def canonical_answer_key(ans: str) -> str:
    ans = clean_answer_candidate(ans)
    val = parse_numeric_answer(ans)
    if val is not None and math.isfinite(float(val)):
        return f"num:{float(val):.10g}"
    return f"str:{ans}"


def choose_answer_from_candidates(cands: List[str]) -> Tuple[Optional[str], Dict[str, Any]]:
    cleaned = [clean_answer_candidate(c) for c in cands if c and clean_answer_candidate(c)]
    if not cleaned:
        return None, {"source": "none", "candidates": []}

    grouped = defaultdict(list)
    for c in cleaned:
        grouped[canonical_answer_key(c)].append(c)

    # Prefer the most common canonical answer; tie-break by shorter representative.
    best_key, best_group = sorted(
        grouped.items(),
        key=lambda kv: (-len(kv[1]), min(len(x) for x in kv[1]), kv[0]),
    )[0]
    representative = sorted(best_group, key=lambda x: (len(x), x))[0]
    return representative, {
        "source": "model_vote",
        "candidates": cleaned,
        "canonical_counts": {k: len(v) for k, v in grouped.items()},
        "best_key": best_key,
    }


def make_model_output(answer: str, source: str = "model") -> str:
    answer = clean_answer_candidate(answer or "")
    if not answer:
        # This is only a last-resort formatting fallback for invalid generations.
        # It is not a solver and does not inspect the query to compute an answer.
        answer = "0"
    return f"Lời giải ngắn: Mô hình dự đoán đáp án từ câu hỏi đã cho.\nĐáp án là: {answer}"


def predict_one(query: str) -> Tuple[str, Dict[str, Any]]:
    prompt = make_prompt(normalize_query(query))

    all_candidates = []
    source_trace = []

    # 1) Deterministic model generation.
    greedy_cands = generate_answer_only(prompt, do_sample=False, num_return_sequences=1)
    all_candidates.extend(greedy_cands)
    source_trace.append({"source": "model_greedy", "candidates": greedy_cands})

    # 2) Optional model-only sampling fallback. Every candidate is still generated by the model.
    if USE_SAMPLING_FALLBACK:
        sample_cands = generate_answer_only(prompt, do_sample=True, num_return_sequences=NUM_SAMPLE_CANDIDATES)
        all_candidates.extend(sample_cands)
        source_trace.append({"source": "model_sample", "candidates": sample_cands})

    ans, info = choose_answer_from_candidates(all_candidates)
    if ans is not None:
        return make_model_output(ans, source="model_vote"), {
            "source": "model_vote",
            "answer": ans,
            "trace": source_trace,
            **info,
        }

    return make_model_output("0", source="fallback_format_only"), {
        "source": "fallback_format_only",
        "answer": "0",
        "trace": source_trace,
    }


## Cell 10 — Validation inference + report

In [10]:

def evaluate_records(records: List[Dict[str, Any]], max_samples: Optional[int] = None) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    eval_records = list(records)
    if max_samples is not None and len(eval_records) > max_samples:
        rng = random.Random(SEED)
        eval_records = rng.sample(eval_records, max_samples)

    outputs = []
    for i, r in enumerate(eval_records):
        if i % 50 == 0:
            print(f"Evaluating {i}/{len(eval_records)}")
        model_output, info = predict_one(r["query_vi"])
        pred_answer = extract_final_answer(model_output, fallback_last_number=True)
        gold_answer = r["final_answer"]
        sc = score_one(pred_answer, gold_answer)
        err = relative_error(pred_answer, gold_answer)
        outputs.append({
            "id": r.get("id", i),
            "raw_index": r.get("raw_index", i),
            "query_vi": r["query_vi"],
            "type": r["type"],
            "gold_answer": gold_answer,
            "pred_answer": clean_answer_candidate(pred_answer or ""),
            "relative_error": err,
            "score": sc,
            "model_output": model_output,
            "predict_info": info,
        })

    n = len(outputs)
    report = {
        "variant": VARIANT_NAME,
        "n": n,
        "raw_score": int(sum(x["score"] for x in outputs)),
        "score_10": float(sum(x["score"] for x in outputs) / n) if n else 0.0,
        "exact_10_count": int(sum(x["score"] == 10 for x in outputs)),
        "score_5_count": int(sum(x["score"] == 5 for x in outputs)),
        "score_1_count": int(sum(x["score"] == 1 for x in outputs)),
        "score_0_count": int(sum(x["score"] == 0 for x in outputs)),
        "extract_rate": float(np.mean([bool(x["pred_answer"]) for x in outputs])) if n else 0.0,
        "source_counts": dict(Counter(x["predict_info"].get("source", "unknown") for x in outputs)),
        "inference_policy": INFERENCE_POLICY,
    }
    return outputs, report

if DO_VALIDATE:
    valid_outputs, valid_report = evaluate_records(valid_records, VALID_EVAL_MAX_SAMPLES)
    write_json(valid_outputs, WORK_DIR / "valid_output.json")
    write_json(valid_report, WORK_DIR / "valid_report.json")
    print(json.dumps(valid_report, ensure_ascii=False, indent=2))
else:
    valid_outputs, valid_report = [], {}
    print("DO_VALIDATE=False, skip validation")


Evaluating 0/997
Evaluating 50/997
Evaluating 100/997
Evaluating 150/997
Evaluating 200/997
Evaluating 250/997
Evaluating 300/997
Evaluating 350/997
Evaluating 400/997
Evaluating 450/997
Evaluating 500/997
Evaluating 550/997
Evaluating 600/997
Evaluating 650/997
Evaluating 700/997
Evaluating 750/997
Evaluating 800/997
Evaluating 850/997
Evaluating 900/997
Evaluating 950/997
{
  "variant": "v11_answer_only_lora_model_only",
  "n": 997,
  "raw_score": 1399,
  "score_10": 1.40320962888666,
  "exact_10_count": 99,
  "score_5_count": 30,
  "score_1_count": 259,
  "score_0_count": 609,
  "extract_rate": 1.0,
  "source_counts": {
    "model_vote": 997
  },
  "inference_policy": {
    "variant": "v11_answer_only_lora_model_only",
    "model_only": true,
    "no_rule_solver": true,
    "no_python_math_solver_from_query": true,
    "allowed_python_usage": [
      "data_loading",
      "data_cleaning",
      "answer_extraction_from_model_output",
      "local_validation_scoring",
      "json_expo

## Cell 11 — Report by type and error table

In [11]:

if valid_outputs:
    df_eval = pd.DataFrame(valid_outputs)

    def summarize_group(g: pd.DataFrame) -> pd.Series:
        return pd.Series({
            "n": len(g),
            "score_10": g["score"].sum() / len(g) if len(g) else 0,
            "extract_rate": (g["pred_answer"].astype(str).str.len() > 0).mean() if len(g) else 0,
            "score_10_count": int((g["score"] == 10).sum()),
            "score_5_count": int((g["score"] == 5).sum()),
            "score_1_count": int((g["score"] == 1).sum()),
            "score_0_count": int((g["score"] == 0).sum()),
        })

    type_report = df_eval.groupby("type").apply(summarize_group).reset_index()
    type_report = type_report.sort_values(["score_10", "n"], ascending=[True, False])
    display(type_report)
    type_report.to_csv(WORK_DIR / "valid_report_by_type.csv", index=False, encoding="utf-8-sig")

    debug_cols = ["raw_index", "type", "score", "relative_error", "gold_answer", "pred_answer", "query_vi", "model_output", "predict_info"]
    errors_df = df_eval[df_eval["score"] < 10].sort_values(["score", "relative_error"], ascending=[True, False], na_position="last")
    display(errors_df[debug_cols].head(30))
    errors_df[debug_cols].to_csv(WORK_DIR / "valid_errors.csv", index=False, encoding="utf-8-sig")

    full_summary = {"overall": valid_report, "by_type": type_report.to_dict(orient="records"), "preprocess": prep_report}
    write_json(full_summary, WORK_DIR / "valid_full_summary.json")
else:
    print("No valid outputs to report.")


,type,n,score_10,extract_rate,score_10_count,score_5_count,score_1_count,score_0_count
4,MATH_AnsAug,171.0,1.040936,1.0,12.0,4.0,38.0,117.0
0,GSM_AnsAug,209.0,1.066986,1.0,15.0,6.0,43.0,145.0
6,MATH_Rephrased,115.0,1.304348,1.0,10.0,4.0,30.0,71.0
2,GSM_Rephrased,197.0,1.416244,1.0,17.0,10.0,59.0,111.0
3,GSM_SV,97.0,1.463918,1.0,11.0,1.0,27.0,58.0
7,MATH_SV,41.0,1.609756,1.0,4.0,2.0,16.0,19.0
1,GSM_FOBAR,122.0,1.877049,1.0,19.0,1.0,34.0,68.0
5,MATH_FOBAR,45.0,2.933333,1.0,11.0,2.0,12.0,20.0


,raw_index,type,score,relative_error,gold_answer,pred_answer,query_vi,model_output,predict_info
801,804,MATH_Rephrased,0,671.666667,3,2018,"Có bao nhiêu bộ số thực $(x,y,z)$ thỏa mãn các...",Lời giải ngắn: Mô hình dự đoán đáp án từ câu h...,"{'source': 'model_vote', 'answer': '2018', 'tr..."
147,147,GSM_AnsAug,0,399.000000,10,4000,Nancy muốn tìm hiểu xem liệu cô ấy có đủ khả n...,Lời giải ngắn: Mô hình dự đoán đáp án từ câu h...,"{'source': 'model_vote', 'answer': '4000', 'tr..."
558,561,GSM_AnsAug,0,89.000000,3,270,Johnny đã viết một bài luận khoảng 150 từ. Mad...,Lời giải ngắn: Mô hình dự đoán đáp án từ câu h...,"{'source': 'model_vote', 'answer': '270', 'tra..."
807,810,MATH_Rephrased,0,33.448276,29,999,Xác định số tự nhiên từ 6 đến 60 (loại trừ) ng...,Lời giải ngắn: Mô hình dự đoán đáp án từ câu h...,"{'source': 'model_vote', 'answer': '999', 'tra..."
440,442,MATH_Rephrased,0,32.333333,3,100,"Cơ số, ký hiệu là b, sao cho số 100 trong cơ s...",Lời giải ngắn: Mô hình dự đoán đáp án từ câu h...,"{'source': 'model_vote', 'answer': '100', 'tra..."
529,532,MATH_AnsAug,0,26.000000,4,-100,Giải $x$: $2(3^x) = 162$.,Lời giải ngắn: Mô hình dự đoán đáp án từ câu h...,"{'source': 'model_vote', 'answer': '-100', 'tr..."
742,745,GSM_AnsAug,0,26.000000,4,108,Clara đang leo lên đỉnh một tòa tháp lịch sử b...,Lời giải ngắn: Mô hình dự đoán đáp án từ câu h...,"{'source': 'model_vote', 'answer': '108', 'tra..."
685,688,GSM_FOBAR,0,24.000000,2,50,Một bà nội trợ đi chợ. Cô ấy đã tiêu x trong s...,Lời giải ngắn: Mô hình dự đoán đáp án từ câu h...,"{'source': 'model_vote', 'answer': '50', 'trac..."
971,974,MATH_AnsAug,0,23.500000,2,-45,Phân tích nhân tử của biểu thức sau: $45x+30$.,Lời giải ngắn: Mô hình dự đoán đáp án từ câu h...,"{'source': 'model_vote', 'answer': '-45', 'tra..."
614,617,GSM_FOBAR,0,20.428571,14,300,Winston có x khu. Sau đó anh ta chi nửa đô la ...,Lời giải ngắn: Mô hình dự đoán đáp án từ câu h...,"{'source': 'model_vote', 'answer': '300', 'tra..."


## Cell 12 — Generate `test_predictions.json` when test exists

In [12]:

def build_test_records(raw_test: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    records = []
    for i, raw in enumerate(raw_test):
        q = normalize_query(str(raw.get("query_vi", "") or ""))
        records.append({
            "id": raw.get("id", i),
            "query_vi": q,
            "type": str(raw.get("type", "UNKNOWN") or "UNKNOWN"),
        })
    return records


def predict_test(records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    preds = []
    for i, r in enumerate(records):
        if i % 50 == 0:
            print(f"Predicting {i}/{len(records)}")
        model_output, info = predict_one(r["query_vi"])
        preds.append({
            "id": r["id"],
            "query_vi": r["query_vi"],
            "type": r["type"],
            "model_output": model_output,
        })
    return preds

if DO_TEST_PREDICT and TEST_PATH is not None and TEST_PATH.exists():
    print("Found test file:", TEST_PATH)
    raw_test = ensure_list_records(read_json_or_jsonl(TEST_PATH))
    test_records = build_test_records(raw_test)
    test_predictions = predict_test(test_records)
    write_json(test_predictions, WORK_DIR / "test_predictions.json")
    print("Saved:", WORK_DIR / "test_predictions.json", "n=", len(test_predictions))
elif DO_TEST_PREDICT:
    # Smoke-test output format using first 2 valid examples so the notebook always leaves an example file.
    print("No official test.json found. Creating sample_test_predictions_from_valid.json for format check only.")
    sample_records = [
        {"id": r["id"], "query_vi": r["query_vi"], "type": r["type"]}
        for r in valid_records[:2]
    ]
    sample_predictions = predict_test(sample_records)
    write_json(sample_predictions, WORK_DIR / "sample_test_predictions_from_valid.json")
    display(pd.DataFrame(sample_predictions))
else:
    print("DO_TEST_PREDICT=False, skip test prediction")


No official test.json found. Creating sample_test_predictions_from_valid.json for format check only.
Predicting 0/2


,id,query_vi,type,model_output
0,0,Nếu Susan đang chơi một trò chơi cờ bàn có 48 ...,GSM_Rephrased,Lời giải ngắn: Mô hình dự đoán đáp án từ câu h...
1,1,"Nếu $\angle PQR = \angle PRQ$, và độ dài của Q...",MATH_Rephrased,Lời giải ngắn: Mô hình dự đoán đáp án từ câu h...


## Cell 13 — Final artifact list

In [13]:

print("Important outputs:")
for p in [
    WORK_DIR / "valid_output.json",
    WORK_DIR / "valid_report.json",
    WORK_DIR / "valid_report_by_type.csv",
    WORK_DIR / "valid_errors.csv",
    WORK_DIR / "test_predictions.json",
    WORK_DIR / "sample_test_predictions_from_valid.json",
    MODEL_OUT_DIR,
]:
    print("-", p, "exists=", p.exists())


Important outputs:
- /kaggle/working/valid_output.json exists= True
- /kaggle/working/valid_report.json exists= True
- /kaggle/working/valid_report_by_type.csv exists= True
- /kaggle/working/valid_errors.csv exists= True
- /kaggle/working/test_predictions.json exists= False
- /kaggle/working/sample_test_predictions_from_valid.json exists= True
- /kaggle/working/lora_v11_answer_only_lora_model_only exists= True
